# Apartment Defect AI — ConvNeXt-Tiny 전체 학습

먼저 Colab 메뉴에서 **런타임 → 런타임 유형 변경 → T4 GPU 이상**을 선택합니다. 이 노트북은 Google Drive의 입력 번들을 풀고, 공개 이미지 URL을 내려받아 학습한 뒤 결과 ZIP을 Drive에 저장합니다. DB 환경변수나 비밀번호는 사용하지 않습니다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
BASE = Path('/content/drive/MyDrive/Apartment_Defect_AI')
BUNDLE = BASE / 'colab-input' / 'defect-convnext-full-colab.zip'
RESULTS = BASE / 'colab-results'
assert BUNDLE.is_file(), f'입력 번들을 찾을 수 없습니다: {BUNDLE}'
RESULTS.mkdir(parents=True, exist_ok=True)

In [ ]:
import shutil, subprocess, sys, torch
WORK = Path('/content/apartment-defect-ai')
if WORK.exists(): shutil.rmtree(WORK)
WORK.mkdir()
shutil.unpack_archive(BUNDLE, WORK)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', f'{WORK}/project[pytorch]'], check=True)
assert torch.cuda.is_available(), 'CUDA GPU가 활성화되지 않았습니다.'
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
DATA = Path('/content/defect-training-data')
if DATA.exists(): shutil.rmtree(DATA)
subprocess.run(['apartment-data', 'vision-colab-prepare', str(WORK), str(DATA), '--workers', '32'], check=True)
print((DATA / 'colab_dataset_manifest.json').read_text())

In [ ]:
RUN = Path('/content/defect-convnext-full-run')
if RUN.exists(): shutil.rmtree(RUN)
subprocess.run([
    'apartment-data', 'vision-train', str(DATA / 'training_spec.json'), str(RUN),
    '--backend', 'pytorch', '--device', 'cuda',
    '--architecture', 'convnext_tiny', '--pretrained',
    '--epochs', '5', '--batch-size', '32', '--learning-rate', '0.0001'
], check=True)
print((RUN / 'final_metrics.json').read_text())

In [ ]:
import hashlib, datetime
stamp = datetime.datetime.now().strftime('%Y%m%d-%H%M%S')
archive_base = RESULTS / f'defect-convnext-full-run-{stamp}'
zip_path = Path(shutil.make_archive(str(archive_base), 'zip', RUN.parent, RUN.name))
digest = hashlib.sha256()
with zip_path.open('rb') as stream:
    for block in iter(lambda: stream.read(1024 * 1024), b''): digest.update(block)
sha256 = digest.hexdigest()
zip_path.with_suffix('.zip.sha256').write_text(f'{sha256}  {zip_path.name}\n')
print('결과:', zip_path)
print('SHA-256:', sha256)